# Ciclo de Vida del Dato en Machine Learning
Este notebook sigue el ciclo de vida del dato para desarrollar un modelo de Machine Learning, desde la definición del problema hasta la monitorización del modelo en producción.

## 1. Definición del Problema
En este proyecto, el objetivo es predecir el gasto total de los clientes de Horizon Digital basado en sus características demográficas y de comportamiento.

### Visión de Negocio
Horizon Digital busca identificar clientes de alto valor y optimizar sus estrategias de marketing.

### Análisis Preliminar
Exploraremos las variables disponibles para entender su distribución y relación con el gasto total.

In [ ]:
# 2. Obtención de Datos
import pandas as pd
import numpy as np
import plotly.express as px

# Cargar el dataset
df = pd.read_csv('ecommerce_data_1M.csv')
df['LastPurchaseDate'] = pd.to_datetime(df['LastPurchaseDate'])

# Vista previa de los datos
df.head()

In [ ]:
# 3. Preparación y Tratamiento de Datos
# Manejo de valores nulos
df = df.dropna()

# Conversión de tipos de datos
df['IsSubscribed'] = df['IsSubscribed'].astype(int)

# Identificación de outliers con boxplot
fig = px.box(df, y='AnnualIncome', title='Outliers en Ingresos Anuales')
fig.show()

# Normalización de datos
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[['Age', 'AnnualIncome', 'TimeOnSite']] = scaler.fit_transform(df[['Age', 'AnnualIncome', 'TimeOnSite']])

In [ ]:
# 4. Dividir los Conjuntos de Datos
from sklearn.model_selection import train_test_split

# Selección de características y variable objetivo
features = ['Age', 'AnnualIncome', 'TimeOnSite', 'ItemsInCart']
X = df[features]
y = df['TotalSpending']

# División en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 5. Construir el Modelo
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Entrenamiento del modelo base
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

# Optimización de hiperparámetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

# Mejor modelo
best_rf = grid_search.best_estimator_
print(f'Mejores hiperparámetros: {grid_search.best_params_}')

In [ ]:
# 6. Análisis de Errores y Reentrenamiento
from sklearn.metrics import mean_squared_error

# Predicciones y evaluación
y_pred = best_rf.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'RMSE en el conjunto de prueba: {rmse:.2f}')

# Análisis de errores
error_df = pd.DataFrame({'Real': y_test, 'Predicción': y_pred, 'Error': y_test - y_pred})
fig = px.scatter(error_df, x='Real', y='Error', title='Análisis de Errores')
fig.show()

# 7. Productivizar e Integrar el Modelo
El modelo optimizado puede ser exportado y utilizado en un sistema de producción para realizar predicciones en tiempo real.

```python
import joblib
joblib.dump(best_rf, 'modelo_random_forest.pkl')
```

# 8. Monitorizar, Mantener y Mejorar el Proceso
Es importante establecer métricas de rendimiento y un sistema de monitorización para evaluar el desempeño del modelo en producción. Esto incluye:
- Seguimiento de métricas como RMSE y MAE.
- Retrain del modelo con nuevos datos.
- Ajustes periódicos de hiperparámetros.